Working on RTX 6000Ada 48GB (per-device batch size 2) and H100 80GB (per-device batch size 16)

In [1]:
# !pip install jiwer
# !pip install evaluate
# !pip install -U accelerate
# !pip install -U torch
# !pip install -U torchvision
# !pip install  transformers[torch]
# !pip install  soundfile
# !git clone https://github.com/jqug/salt.git
# !pip install -r salt/requirements.txt
# !pip install  peft

In [2]:
use_wandb = False
use_mlflow = True

import importlib.metadata
installed = [
    dist.metadata['Name']
    for dist in importlib.metadata.distributions()
]

if use_wandb:
  !pip install  wandb
  import wandb
  %set_env WANDB_LOG_MODEL=True
  %set_env WANDB_WATCH=all
  %set_env WANDB_NOTEBOOK_NAME=whisper_base_en_sb.ipynb
  wandb.login()

if use_mlflow:
  if 'mlflow' not in installed:
      !pip install  mlflow
      ## requirements to log system/GPU metrics in mlflow
  !pip install  psutil
  !pip install  pynvml
  import os
  from getpass import getpass
  import mlflow
  import mlflow.pytorch
  from mlflow import MlflowClient

  # Set MLflow tracking credentials
  MLFLOW_TRACKING_USERNAME = getpass('Enter the MLFLOW_TRACKING_USERNAME: ')
  os.environ['MLFLOW_TRACKING_USERNAME'] = MLFLOW_TRACKING_USERNAME

  MLFLOW_TRACKING_PASSWORD = getpass('Enter the MLFLOW_TRACKING_PASSWORD: ')
  os.environ['MLFLOW_TRACKING_PASSWORD'] = MLFLOW_TRACKING_PASSWORD

  # Set the MLflow tracking URI
  mlflow.set_tracking_uri('https://mlflow-sunbird-ce0ecfc14244.herokuapp.com/')
  mlflow.system_metrics.enable_system_metrics_logging()

Enter the MLFLOW_TRACKING_USERNAME:  ········
Enter the MLFLOW_TRACKING_PASSWORD:  ········


In [3]:
import sys
from pathlib import Path
root_dir = Path.cwd().parent.parent.parent
sys.path.append(str(root_dir))

import torch
import transformers
from dataclasses import dataclass, field
from typing import Union, List, Dict, Any
import string
import os
import json
import datasets
import numpy as np
import yaml
import evaluate
import salt.dataset
import salt.metrics
import salt.constants
from salt.utils import DataCollatorCTCWithPadding as dcwp
import huggingface_hub
import peft
import pandas as pd
import gcsfs

In [4]:
# huggingface_hub.notebook_login()

In [5]:
# import gcsfs
# gcs = gcsfs.GCSFileSystem(project='sb-gcp-project-01', token='google_default')
# bucket_path = "sunflower-data/speech"
# base_gcs = f"gcs://{bucket_path}"
# all_datasets = gcs.ls(path=bucket_path, detail=False)
# all_datasets

In [6]:
with open('whisper_finetuning_gcp.yaml', 'r') as file:
    config = yaml.safe_load(file)

In [7]:
train_ds = salt.dataset.create(config['train'], verbose=False)
valid_ds = salt.dataset.create(config['validation'])

In [8]:
for i, example in enumerate(train_ds.take(1)):
    print(f"--- Example {i} ---")
    
    # Check if 'source' or 'target' contains large audio arrays
    # If they are dicts with 'array' keys, print just the shape/metadata
    for key in ['source', 'target']:
        data = example.get(key)
        if isinstance(data, dict) and 'array' in data:
            # Avoid printing the massive raw audio array
            print(f"{key}: [Audio Data] Sample Rate: {data.get('sampling_rate')}, Shape: {data['array'].shape}")
        else:
            print(f"{key}: {data}")
            
    print(f"Languages: {example.get('source.language')} -> {example.get('target.language')}\n")

--- Example 0 ---
source: [ 2.0106706e-06  3.2367443e-06 -2.7039619e-06 ... -1.4706944e-06
  2.6853128e-05  1.4958448e-05]
target: Ikiraro cya gari ya moshi kimaze kubakwa hejuru yuruzi. Ejo bundi nasanze igitabo cyanditswe na data. Ugomba guhindura uruhushya rwawe rwa kera kugirango ubone urundi rushya. Yemeza mushiki wa Denis, Ndayishimiye yakomeje avuga ko hari ingaruka nyinshi. Maze bigafasha mwarimu uburyo yatangamo isomo rye neza.
Languages: lug -> lug



In [9]:
salt.utils.show_dataset(train_ds, audio_features=['source'], N=1)

,source,target,source.language,target.language
0,Your browser does not support the audio element.,"Hazaba ikindi cyerekanwa cyiyi firime mumasaha abiri. Shimirwa nyagasani, imbaraga zawe. David ashobora gufungwa burundu. Akajya ayatekereza. Maze abungeri basubirayo bahimbaza, bashima Imana.",lug,lug


In [25]:
feature_extractor = transformers.WhisperFeatureExtractor.from_pretrained(
    config['pretrained_model'])
processor = transformers.WhisperProcessor.from_pretrained(
    config['pretrained_model'], language=None, task="transcribe")
model = transformers.WhisperForConditionalGeneration.from_pretrained(
    config['pretrained_model'], torch_dtype=torch.float32)

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

In [26]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lengths and need different padding methods
        # first treat the audio inputs by simply returning torch tensors
        input_features = [{"input_features": feature["input_features"]} for feature in features]    
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # get the tokenized label sequences
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        # pad the labels to max length
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # if bos token is appended in previous tokenization step,
        # cut bos token here as it's append later anyways
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor, decoder_start_token_id=model.config.decoder_start_token_id)

Read in prompts: preceding text which is used to guide the model.

In [27]:
# sentences = datasets.load_dataset(
#     'Sunbird/salt', 'text-all', split='train').to_pandas()
# prompts = datasets.load_dataset(
#     'Sunbird/prompts', split='train').to_pandas()
# joined = pd.merge(sentences, prompts, on='id', how='inner')
# SALT_PROMPT_LANGUAGES = ['eng', 'ach', 'lgg', 'lug', 'nyn', 'teo']
# sentence_to_prompt = {}
# for language in SALT_PROMPT_LANGUAGES:
#     sentence_to_prompt[language] = dict(
#         zip(joined[f'{language}_text'], joined[f'{language}_prompt']))

In [28]:
language_id_tokens = salt.constants.SALT_LANGUAGE_TOKENS_WHISPER

def prepare_dataset(example, p_prompt = 0.5):    
    audio = example["source"]
    input_features = feature_extractor(
        audio, sampling_rate=16000, device='cuda',
        do_normalize=True).input_features[0]

    # Encode target text to label ids
    labels = processor.tokenizer(str(example["target"])).input_ids

    # Insert the language ID token into the second position of the sequence.
    labels.insert(1, language_id_tokens[example["target.language"]])

    # # If a prompt is known for a particular sentence, add it to the
    # # training example with probability `p_prompt`.
    # if example["target.language"] in sentence_to_prompt:
    #     prompt = sentence_to_prompt[example["target.language"]].get(example["target"], None)
    #     if prompt:
    #         if np.random.random() < p_prompt:
    #             prompt_ids = list(processor.get_prompt_ids(prompt))
    #             labels = prompt_ids + labels  

    # Create a new dictionary with the processed data
    processed_example = {
        "input_features": input_features,
        "labels": np.array(labels),
        "source.language": example["source.language"],
        "target.language": example["target.language"]
    }

    return processed_example

In [29]:
train_data = train_ds.map(prepare_dataset, remove_columns=["source", "target"])
val_data = valid_ds.map(prepare_dataset, remove_columns=["source", "target"])

In [30]:
compute_metrics = salt.metrics.multilingual_eval_fn(
      valid_ds, [evaluate.load('wer'), evaluate.load('cer')],
      processor.tokenizer, log_first_N_predictions=5,
      speech_processor=processor)

In [31]:
from transformers import GenerationConfig

gen_config = GenerationConfig.from_pretrained(config['pretrained_model'])
gen_config.suppress_tokens = []  # This clears the default suppressed tokens
gen_config.max_length = 200     # Matches your 'generation_max_length'
gen_config.forced_decoder_ids = None
model.generation_config = gen_config

model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.eos_token_id = processor.tokenizer.eos_token_id

if config['use_peft']:
    model = peft.prepare_model_for_kbit_training(model)
    lora_config = peft.LoraConfig(**config['lora_config'])
    model.enable_input_require_grads()
    model = peft.get_peft_model(model, lora_config)
    model.config.use_cache = False
    model.print_trainable_parameters()

In [32]:
config["training_args"]

{'output_dir': 'whisper-large-v3-multilingual',
 'per_device_train_batch_size': 2,
 'per_device_eval_batch_size': 2,
 'gradient_accumulation_steps': 4,
 'learning_rate': 1e-05,
 'warmup_steps': 500,
 'max_steps': 7500,
 'gradient_checkpointing': False,
 'gradient_checkpointing_kwargs': {'use_reentrant': False},
 'fp16': False,
 'bf16': True,
 'eval_strategy': 'steps',
 'predict_with_generate': True,
 'generation_max_length': 200,
 'save_steps': 50,
 'eval_steps': 50,
 'logging_steps': 50,
 'load_best_model_at_end': True,
 'metric_for_best_model': 'loss',
 'greater_is_better': False,
 'push_to_hub': True,
 'hub_model_id': 'jq/whisper-large-v3-salt-plus-xog-myx-kin-swa',
 'save_total_limit': 2}

Launch the training

In [ ]:
training_args = transformers.Seq2SeqTrainingArguments(
  **config["training_args"],
  report_to= [
      platform for platform, use in [("wandb", use_wandb), ("mlflow", use_mlflow)] if use]
)

trainer = transformers.Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_data,
    eval_dataset=val_data,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor,
)

trainer.train()

2026/03/05 08:00:39 ERROR mlflow.utils.async_logging.async_logging_queue: Run Id 2d488acdc39146e9af9da07c00128d49: Failed to log run data: Exception: INVALID_PARAMETER_VALUE: Changing param values is not allowed. Params were already logged='[{'key': 'dtype', 'old_value': 'float16', 'new_value': 'float32'}]' for run ID='2d488acdc39146e9af9da07c00128d49'.


Step,Training Loss,Validation Loss


A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.


In [ ]:
%debug

Log the config settings for reference

In [ ]:
if use_mlflow:
    mlflow.log_params(config)

Save the full model (not just the adapter weights)

In [ ]:
processor.push_to_hub(config['training_args']['hub_model_id'])
model.push_to_hub(config['training_args']['hub_model_id'])

Try running the model on the first test example

In [ ]:
example = next(iter(valid_ds))
input_features = processor(example["source"], sampling_rate=16000, return_tensors="pt").input_features
with torch.no_grad():
    predicted_ids = model.generate(input_features.to("cuda"))[0]
transcription = processor.decode(predicted_ids)
print(transcription)